# Activity 3: Capstone Survival Workshop — Bug Hunt & Defense Gauntlet
**STIWNU Computing Seminar • November 23, 2026**  
**Runtime:** Default **CPU**

### Workshop Goals
1. **Part A (Bug Hunt):** Spot **4 fatal capstone-killer bugs** hidden inside a realistic Barangay Flood Risk Predictor.
2. **Part B (The Fix):** Compare the broken model vs. the fixed pipeline side-by-side and see why **Accuracy lies** on rare disasters!
3. **Part C (Responsible 2026 Colab AI):** Practice writing engineering prompts instead of blind copy-pasting.
4. **Part D (Capstone Pre-Defense Checklist):** Self-audit your own thesis idea before the Q&A clinic.

## Part A: CHALLENGE — Find the 4 Capstone-Killer Bugs!
**Instructions (5 Minutes in Pairs):** Read and run the broken Barangay Flood Predictor below. Notice it prints **~95% Accuracy**! Can you identify ALL 4 engineering bugs hiding inside these 20 lines of code before scrolling down?

In [1]:
# ==============================================================================
# PART A: BROKEN BARANGAY FLOOD PREDICTOR — SPOT THE 4 FATAL BUGS!
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Simulated 1,000-day Barangay Sensor Dataset (95% No Flood = 0, 5% Severe Flood = 1)
X, y = make_classification(
    n_samples=1000, n_features=4, n_informative=3, n_redundant=0,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
feature_names = ['Rainfall_mm', 'River_Level_m', 'Soil_Moisture', 'Tide_Height_m']

# ---> LOOK CLOSELY AT THE 4 LINES BELOW! <---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)                                  # ???

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42                    # ???
)

broken_model = RandomForestClassifier(random_state=42)              # ???
broken_model.fit(X_train, y_train)

broken_preds = broken_model.predict(X_test)
print(f'Broken Model Reported Accuracy: {accuracy_score(y_test, broken_preds)*100:.2f}%')  # ???

Broken Model Reported Accuracy: 97.50%


---
## Part B: STOP! Only Run Below After Guessing All 4 Bugs
Let's expose what that '95% accurate' broken model was hiding—and compare it directly against a clean, leakage-free, class-weighted pipeline!

In [ ]:
# ==============================================================================
# PART B: EXPOSING THE BUGS & RUNNING THE CLEAN CAPSTONE-READY FIX
# ==============================================================================
from sklearn.pipeline import Pipeline

print('=== 1. WHAT THE BROKEN MODEL ACTUALLY HID ===')
print(classification_report(y_test, broken_preds, target_names=['No Flood (0)', 'Severe Flood (1)']))

# FIX 1 & 2: Split RAW data FIRST with stratify=y (Zero Leakage + Balanced Splits!)
X_tr_clean, X_te_clean, y_tr_clean, y_te_clean = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# FIX 3: Put StandardScaler INSIDE a Pipeline + add class_weight='balanced'
clean_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',   # Penalizes missing rare floods heavily!
        max_depth=6,
        random_state=42
    ))
])

clean_pipeline.fit(X_tr_clean, y_tr_clean)

# Optional DRRM Safety Tuning: Lower alert threshold from 0.50 -> 0.30 to maximize Flood Recall
flood_probs = clean_pipeline.predict_proba(X_te_clean)[:, 1]
clean_preds = (flood_probs >= 0.30).astype(int)

# FIX 4: Report Precision, Recall, F1-Score & Confusion Matrix (Never Accuracy Alone!)
print('=== 2. CLEAN, LEAKAGE-FREE DRRM PIPELINE (SAFETY THRESHOLD = 0.30) ===')
print(classification_report(y_te_clean, clean_preds, target_names=['No Flood (0)', 'Severe Flood (1)']))
print('Confusion Matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_te_clean, clean_preds))

## Part C: Responsible 2026 Colab AI Prompting Playground
Colab's built-in Gemini / Data Science Agent is a superpower—**if** you prompt it like an engineer instead of blindly copy-pasting black-box code you cannot explain to panelists.

| Lazy Prompt (Fails Defense) | Engineer Prompt (Wins Defense) |
| :--- | :--- |
| *"Make my flood prediction capstone model."* | *"Inspect `flood_data.csv` for nulls, duplicates, and class imbalance. Do NOT edit yet. Propose a leakage-free preprocessing plan and explain why you chose each step."* |
| *"Fix my accuracy to 99%."* | *"My validation loss is rising while training loss drops. Diagnose whether this is overfitting or leakage, and suggest 2 regularization fixes with pros/cons."* |

---

## Part D: Interactive Capstone Pre-Defense Self-Check
Edit the variables in the cell below for **your own capstone team** and run it to generate a quick Architecture & Ethics readiness audit before our 4:30 PM Q&A clinic!

In [ ]:
# ==============================================================================
# PART D: CAPSTONE READINESS SELF-AUDIT (EDIT FOR YOUR OWN GROUP!)
# ==============================================================================
capstone_audit = {
    'project_title': 'Smart Barangay Flood & Rice Blight Monitor',
    'data_type': 'vision',            # Options: 'tabular', 'vision', 'both'
    'sample_count_per_class': 250,    # How many rows/photos per class?
    'uses_transfer_learning': True,   # True for vision models with <10k images
    'split_before_transform': True,   # Split BEFORE scaling/SMOTE/augmentation?
    'has_personal_data_or_faces': False,
    'exif_stripped_and_consented': True,
    'primary_metric': 'F1-Score & Recall'
}

print(f"=== CAPSTONE AUDIT: {capstone_audit['project_title']} ===")
warnings = []
if capstone_audit['data_type'] in ('vision', 'both') and capstone_audit['sample_count_per_class'] < 1000 and not capstone_audit['uses_transfer_learning']:
    warnings.append('RISK: Use Transfer Learning (MobileNetV2/YOLO) instead of training from scratch!')
if not capstone_audit['split_before_transform']:
    warnings.append('CRITICAL: Data Leakage detected! Always split train/val/test BEFORE scaling or augmenting.')
if capstone_audit['has_personal_data_or_faces'] and not capstone_audit['exif_stripped_and_consented']:
    warnings.append('PRIVACY RISK (RA 10173 / NPC 2026-01): Blur faces, strip EXIF GPS, and secure signed consent!')
if capstone_audit['primary_metric'].lower() == 'accuracy':
    warnings.append('METRIC WARNING: Pair Accuracy with Precision, Recall, F1-Score, and Confusion Matrix!')

if not warnings:
    print('PASS! Your capstone design covers data hygiene, honest validation, and Philippine privacy compliance.')
else:
    print('ACTION ITEMS BEFORE DEFENSE:')
    for w in warnings:
        print(' -', w)